In [ ]:
import pandas as pd
import numpy as np

def extract_perfect_paper_examples(csv_paths, top_k=3, sim_tolerance=0.01, is_toxic=False, sae_threshold=0.5):
    dfs = [pd.read_csv(p) for p in csv_paths]
    
    base_cols = [
        'category', 'standard_prompt', 'dialect_prompt', 
        'target_sim', 'standard_prompt_NSFW_T_prob', 'dialect_prompt_NSFW_T_prob'
    ]
    merged_df = dfs[0][base_cols].copy()
    
    for i, df in enumerate(dfs):
        merged_df[f'typo_prompt_{i}'] = df['typo_prompt']
        merged_df[f'typo_sim_{i}'] = df['typo_sim']
        merged_df[f'typo_score_{i}'] = df['typo_prompt_NSFW_T_prob']
        
    typo_sim_cols = [f'typo_sim_{i}' for i in range(len(dfs))]
    typo_score_cols = [f'typo_score_{i}' for i in range(len(dfs))]
    
    merged_df['avg_typo_sim'] = merged_df[typo_sim_cols].mean(axis=1)
    merged_df['avg_typo_score'] = merged_df[typo_score_cols].mean(axis=1)
    
    merged_df['sim_diff'] = (merged_df['target_sim'] - merged_df['avg_typo_sim']).abs()
    
    # Calculate Bias Gap (difference between dialect score and typo score)
    merged_df['bias_gap'] = merged_df['dialect_prompt_NSFW_T_prob'] - merged_df['avg_typo_score']
    
    merged_df['prompt_length'] = merged_df['standard_prompt'].apply(lambda x: len(str(x).split()))
    
    def get_best_seed_info(row):
        diffs = [abs(row['target_sim'] - row[col]) for col in typo_sim_cols]
        best_idx = np.argmin(diffs)
        
        return pd.Series({
            'best_typo_prompt': row[f'typo_prompt_{best_idx}'],
            'best_typo_sim': row[f'typo_sim_{best_idx}'],
            'best_sim_diff': diffs[best_idx]
        })
    
    merged_df[['best_typo_prompt', 'best_typo_sim', 'best_sim_diff']] = merged_df.apply(get_best_seed_info, axis=1)
    
    mask = (
        (merged_df['sim_diff'] <= sim_tolerance) &    
        (merged_df['best_sim_diff'] <= sim_tolerance) & 
        (merged_df['prompt_length'] <= 60) &
        (merged_df['target_sim'] > 0.75)
    )
    
    if is_toxic:
        # For Toxic: SAE prompt should be blocked by the filter, so its score must be high.
        mask = mask & (merged_df['standard_prompt_NSFW_T_prob'] >= sae_threshold)
    else:
        # For Benign: Find examples where dialect scores are extremely high (over-censorship)
        mask = mask & (merged_df['dialect_prompt_NSFW_T_prob'] > 0.6)
    
    filtered_df = merged_df[mask].copy()
    
    # Sort: For Toxic, we want to see large absolute differences in bias gap.
    filtered_df['abs_bias_gap'] = filtered_df['bias_gap'].abs()
    
    best_examples = filtered_df.sort_values(
        by=['abs_bias_gap', 'best_sim_diff'], 
        ascending=[False, True]
    ).head(top_k)
    
    return best_examples

In [ ]:
toxic_aave_files = [
    './text_level_typo_results/toxic_results_AAVE_seed_0.csv', 
    './text_level_typo_results/toxic_results_AAVE_seed_1.csv', 
    './text_level_typo_results/toxic_results_AAVE_seed_2.csv',
     './text_level_typo_results/toxic_results_AAVE_seed_3.csv',
      './text_level_typo_results/toxic_results_AAVE_seed_4.csv'
]

best_df = extract_perfect_paper_examples(toxic_aave_files, top_k=2, sim_tolerance=0.01, is_toxic = True)

for idx, row in best_df.iterrows():
    print(f"[{row['category']}] Bias Gap: +{row['bias_gap']:.3f} | Sim Diff: {row['sim_diff']:.3f}")
    
    # SAE (Standard) is the baseline, so sim is exactly 1.000
    sae_score = row.get('standard_prompt_NSFW_T_prob', 0.0)
    print(f"[SAE]      Sim: 1.000 | Score: {sae_score:.3f}")
    print(f"           -> {row['standard_prompt']}")
    
    # Dialect: check how much it was falsely penalized as toxic.
    dial_score = row.get('dialect_prompt_NSFW_T_prob', row.get('score_dialect', 0.0))
    print(f"[Dialect]  Sim: {row['target_sim']:.3f} | Score: {dial_score:.3f}")
    print(f"           -> {row['dialect_prompt']}")
    
    # Typo: check how safe the score is given it has the same distance from standard as dialect.
    typo_prompt = row.get('typo_prompt_1', row.get('typo_prompt_0', ''))
    typo_sim = row.get('typo_sim_1', row.get('typo_sim_0', 0.0))
    typo_score = row.get('typo_score_1', row.get('avg_typo_score', 0.0))
    
    print(f"[Typo(S1)] Sim: {typo_sim:.3f} | Score: {typo_score:.3f}")
    print(f"           -> {typo_prompt}")
    
    print("-" * 70)

In [ ]:
benign_chce_files = [
    './text_level_typo_results/benign_results_ChcE_seed_0.csv', 
    './text_level_typo_results/benign_results_ChcE_seed_1.csv', 
    './text_level_typo_results/benign_results_ChcE_seed_2.csv',
     './text_level_typo_results/benign_results_ChcE_seed_3.csv',
      './text_level_typo_results/benign_results_ChcE_seed_4.csv'
]

best_df = extract_perfect_paper_examples(benign_chce_files, top_k=2, sim_tolerance=0.01, is_toxic = False)

for idx, row in best_df.iterrows():
    print(f"[{row['category']}] Bias Gap: +{row['bias_gap']:.3f} | Sim Diff: {row['sim_diff']:.3f}")
    
    # SAE (Standard) is the baseline, so sim is exactly 1.000
    sae_score = row.get('standard_prompt_NSFW_T_prob', 0.0)
    print(f"[SAE]      Sim: 1.000 | Score: {sae_score:.3f}")
    print(f"           -> {row['standard_prompt']}")
    
    # Dialect: check how much it was falsely penalized as toxic.
    dial_score = row.get('dialect_prompt_NSFW_T_prob', row.get('score_dialect', 0.0))
    print(f"[Dialect]  Sim: {row['target_sim']:.3f} | Score: {dial_score:.3f}")
    print(f"           -> {row['dialect_prompt']}")
    
    # Typo: check how safe the score is given it has the same distance from standard as dialect.
    typo_prompt = row.get('typo_prompt_1', row.get('typo_prompt_0', ''))
    typo_sim = row.get('typo_sim_1', row.get('typo_sim_0', 0.0))
    typo_score = row.get('typo_score_1', row.get('avg_typo_score', 0.0))
    
    print(f"[Typo(S1)] Sim: {typo_sim:.3f} | Score: {typo_score:.3f}")
    print(f"           -> {typo_prompt}")
    
    print("-" * 70)

In [ ]:
import numpy as np
import pandas as pd

def generate_appendix_latex_from_files(csv_paths, dialect="AAVE", mode="Toxic", top_k=5, sim_tolerance=0.015, sae_threshold=0.5):
    dfs = [pd.read_csv(p) for p in csv_paths]
    
    base_cols = [
        'category', 'standard_prompt', 'dialect_prompt', 
        'target_sim', 'standard_prompt_NSFW_T_prob', 'dialect_prompt_NSFW_T_prob'
    ]
    merged_df = dfs[0][base_cols].copy()
    
    for i, df in enumerate(dfs):
        merged_df[f'typo_prompt_{i}'] = df['typo_prompt']
        merged_df[f'typo_sim_{i}'] = df['typo_sim']
        merged_df[f'typo_score_{i}'] = df['typo_prompt_NSFW_T_prob']
        
    typo_sim_cols = [f'typo_sim_{i}' for i in range(len(dfs))]
    typo_score_cols = [f'typo_score_{i}' for i in range(len(dfs))]
    
    sim_diffs = pd.DataFrame({
        f'd{i}': (merged_df['target_sim'] - merged_df[typo_sim_cols[i]]).abs() 
        for i in range(len(dfs))
    })
    merged_df['max_sim_diff'] = sim_diffs.max(axis=1)
    
    merged_df['max_typo_score'] = merged_df[typo_score_cols].max(axis=1)
    merged_df['avg_typo_score'] = merged_df[typo_score_cols].mean(axis=1)
    
    merged_df['bias_gap'] = merged_df['dialect_prompt_NSFW_T_prob'] - merged_df['avg_typo_score']
    
    condition = (
        (merged_df['max_sim_diff'] <= sim_tolerance) &       
        (merged_df['target_sim'] > 0.8)                       
    )
    
    if mode == "Benign":
        # Benign: Safe in SAE, falsely high in Dialect, safe in Typo.
        condition = condition & (
            (merged_df['standard_prompt_NSFW_T_prob'] < sae_threshold) &
            (merged_df['dialect_prompt_NSFW_T_prob'] > 0.6) &
            (merged_df['max_typo_score'] < 0.5)
        )
    elif mode == "Toxic":
        # Toxic: High in SAE (properly blocked).
        condition = condition & (
            (merged_df['standard_prompt_NSFW_T_prob'] >= sae_threshold)
        )
        
    filtered_df = merged_df[condition].copy()
    
    filtered_df['abs_bias_gap'] = filtered_df['bias_gap'].abs()
    best_df = filtered_df.sort_values(by=['abs_bias_gap', 'max_sim_diff'], ascending=[False, True]).head(top_k)
    
    if best_df.empty:
        return []
    
    def clean_text(text):
        text = str(text).replace("&", "\\&").replace("%", "\\%")
        words = text.split()
        return " ".join(words[:24]) + "..." if len(words) > 12 else text

    lst = []
    for idx in range(len(best_df)):
        
        row = best_df.iloc[idx]
        
        raw_category = str(row['category'])
        if raw_category.lower().startswith("benign_"):
            raw_category = raw_category[7:]
        if raw_category.lower().startswith("toxic_"):
            raw_category = raw_category[6:]
            
        clean_category = raw_category.replace("_", " ").title()
    
        latex_str = f"""
    \\midrule
    \\multirow{{5}}{{*}}{{\\textit{{{mode} ({clean_category})}}}} & \\textbf{{SAE (Base)}} & ``{clean_text(row['standard_prompt'])}'' & 1.000 & {row['standard_prompt_NSFW_T_prob']:.3f} \\\\
    & \\textbf{{{dialect}}} & ``{clean_text(row['dialect_prompt'])}'' & {row['target_sim']:.3f} & {row['dialect_prompt_NSFW_T_prob']:.3f} \\\\
    \\cmidrule{{2-5}}
    & \\textbf{{Typo (Seed 0)}} & ``{clean_text(row['typo_prompt_0'])}'' & {row['typo_sim_0']:.3f} & {row['typo_score_0']:.3f} \\\\
    & \\textbf{{Typo (Seed 1)}} & ``{clean_text(row['typo_prompt_1'])}'' & {row['typo_sim_1']:.3f} & {row['typo_score_1']:.3f} \\\\
    & \\textbf{{Typo (Seed 2)}} & ``{clean_text(row['typo_prompt_2'])}'' & {row['typo_sim_2']:.3f} & {row['typo_score_2']:.3f} \\\\
    & \\textbf{{Typo (Seed 3)}} & ``{clean_text(row['typo_prompt_3'])}'' & {row['typo_sim_3']:.3f} & {row['typo_score_3']:.3f} \\\\
    & \\textbf{{Typo (Seed 4)}} & ``{clean_text(row['typo_prompt_4'])}'' & {row['typo_sim_4']:.3f} & {row['typo_score_4']:.3f} \\\\
        
"""
        lst.append(latex_str)

    return lst

In [ ]:
for candidate in generate_appendix_latex_from_files(toxic_aave_files, top_k=4,dialect="AAVE", mode="Toxic", sim_tolerance=0.05):
    print(candidate)
    print()


In [ ]:
for candidate in generate_appendix_latex_from_files(benign_chce_files, dialect="ChcE", mode="Benign", sim_tolerance=0.03):
    print(candidate)
    print()
